Chatbot Evaluation

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("LANGSMITH_API_KEY")
if api_key:
    os.environ["LANGSMITH_API_KEY"] = api_key

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

os.environ["LANGSMITH_TRACKING"] = "True"

In [2]:
from langsmith import Client
from langsmith.utils import LangSmithError

client = Client()

DATASET_NAME = "RAG-Evaluation"

# Create dataset if it doesn't exist
try:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Dataset for evaluating openai responses."
    )
    print(f"Created dataset: {dataset.name}")

except LangSmithError:
    dataset = client.read_dataset(
        dataset_name=DATASET_NAME
    )
    print(f"Using existing dataset: {dataset.name}")

print("Dataset ID:", dataset.id)


# Dataset examples
inputs = [
    {"question": "What is Gemini?"},
    {"question": "What is LangSmith?"},
    {"question": "What is LangChain?"},
    {"question": "What is the purpose of this dataset?"}
]

outputs = [
    {
        "answer": "Gemini is a family of large language models developed by Google DeepMind."
    },
    {                   
        "answer": "LangSmith is a platform for debugging, testing, evaluating, and monitoring LLM applications."
    },
    {
        "answer": "LangChain is a framework for building applications powered by large language models."
    },
    {
        "answer": "This dataset is used to evaluate the quality and correctness of responses generated by a language model."
    }
]


# Prevent duplicate uploads
existing_examples = list(
    client.list_examples(
        dataset_id=dataset.id
    )
)

if len(existing_examples) == 0:
    client.create_examples(
        inputs=inputs,
        outputs=outputs,
        dataset_id=dataset.id
    )
    print("Examples uploaded successfully!")
else:
    print(
        f"Dataset already contains {len(existing_examples)} examples. Skipping upload."
    )

Using existing dataset: RAG-Evaluation
Dataset ID: da61ae18-c264-43bc-a50e-3d329072f2eb
Dataset already contains 8 examples. Skipping upload.


LLM As a Judge

In [3]:
from openai import OpenAI, OpenAIError, RateLimitError
import os
import re
import threading
import time


# ============================================================
# OpenAI Client
# ============================================================

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

OPENAI_MODEL = "gpt-5-mini"
OPENAI_MIN_SECONDS_BETWEEN_CALLS = 13.0

_openai_lock = threading.Lock()
_last_openai_call_at = 0.0


# ============================================================
# Retry Helper
# ============================================================

def _retry_delay_seconds(
    error: Exception,
    default: float = 60.0,
):
    text = str(error)

    match = re.search(
        r"Please try again in ([\d.]+)s",
        text,
    )

    if not match:
        return default

    return (
        float(
            next(
                x
                for x in match.groups()
                if x is not None
            )
        )
        + 2.0
    )


# ============================================================
# OpenAI Generation Function
# ============================================================

def generate_openai_content(
    model: str,
    contents: str,
    max_retries: int = 5,
):
    global _last_openai_call_at

    with _openai_lock:
        for attempt in range(max_retries):

            elapsed = (
                time.monotonic()
                - _last_openai_call_at
            )

            wait_seconds = (
                OPENAI_MIN_SECONDS_BETWEEN_CALLS
                - elapsed
            )

            if wait_seconds > 0:
                time.sleep(wait_seconds)

            try:
                response = (
                    openai_client.chat.completions.create(
                        model=model,
                        messages=[
                            {
                                "role": "user",
                                "content": contents,
                            }
                        ],
                    )
                )

                _last_openai_call_at = (
                    time.monotonic()
                )

                return (
                    response.choices[0]
                    .message.content
                )

            except (
                RateLimitError,
                OpenAIError,
            ) as error:

                _last_openai_call_at = (
                    time.monotonic()
                )

                error_text = str(error)
                print(f"OpenAI Error:\n{error_text}")

                if (
                    "429" in error_text
                    or "rate_limit" in error_text.lower()
                ):
                    sleep_for = _retry_delay_seconds(
                        error,
                        default=35,
                    )

                    print(
                        f"Retrying in "
                        f"{sleep_for:.0f} seconds..."
                    )

                    time.sleep(sleep_for)
                    continue

                raise

    raise RuntimeError(
        "OpenAI failed after maximum retries."
    )


# ============================================================
# Evaluation Instruction
# ============================================================

instruction = """
Return only True or False.

True:
The model answer is factually correct and
semantically equivalent to the reference answer.

False:
Otherwise.
"""


# ============================================================
# Correctness Evaluator
# ============================================================

def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
):
    question = inputs["question"]

    reference_answer = (
        reference_outputs["answer"]
    )

    model_answer = (
        outputs["answer"]
    )

    user_content = (
        f"Q:{question}\n"
        f"R:{reference_answer}\n"
        f"A:{model_answer}"
    )

    response = generate_openai_content(
        model=OPENAI_MODEL,
        contents=f"{instruction}\n{user_content}",
    )

    is_correct = (
        response.strip().lower()
        == "true"
    )

    return {
        "key": "correctness",
        "score": (
            1.0 if is_correct else 0.0
        ),
    }

In [4]:
#Concision

def concision(
    outputs: dict,
    reference_outputs: dict,
):
    model_answer = outputs["answer"]
    reference_answer = reference_outputs["answer"]

    is_concise = (
        len(model_answer)
        <= 2 * len(reference_answer)
    )

    return {
        "key": "concision",
        "score": (
            1.0 if is_concise else 0.0
        ),
    }

In [5]:
default_instruction = (
    "Answer the question. If unknown, say 'I don't know'."
)

def my_app(
    question: str,
    model: str = OPENAI_MODEL,
    instruction: str = default_instruction,
) -> str:
    response = generate_openai_content(
        model=model,
        contents=f"{instruction}\n{question}"
    )
    return response

In [6]:
_cache = {}

def ls_target(inputs: dict) -> dict:
    question = inputs["question"]

    if question in _cache:
        return _cache[question]

    result = {
        "answer": my_app(question)
    }

    _cache[question] = result
    return result

In [7]:
import time
import re
import threading
from openai import OpenAIError, RateLimitError

OPENAI_MIN_SECONDS_BETWEEN_CALLS = 13.0

_openai_lock = threading.Lock()
_last_openai_call_at = 0.0

# Cache to avoid repeated API calls
_openai_cache = {}


def _retry_delay_seconds(
    error,
    default=60,
):
    text = str(error)

    match = re.search(
        r"Please try again in ([\d.]+)s",
        text,
    )

    if match:
        return (
            float(
                next(
                    x
                    for x in match.groups()
                    if x is not None
                )
            )
            + 2
        )

    return default


def generate_openai_content(
    model: str,
    contents: str,
    max_retries: int = 5,
):
    global _last_openai_call_at

    # Use cached response if available
    cache_key = (model, contents)
    if cache_key in _openai_cache:
        return _openai_cache[cache_key]

    with _openai_lock:

        for attempt in range(max_retries):

            elapsed = (
                time.monotonic()
                - _last_openai_call_at
            )

            wait_seconds = (
                OPENAI_MIN_SECONDS_BETWEEN_CALLS
                - elapsed
            )

            if wait_seconds > 0:
                time.sleep(wait_seconds)

            try:
                response = (
                    openai_client.chat.completions.create(
                        model=model,
                        messages=[
                            {
                                "role": "user",
                                "content": contents,
                            }
                        ],
                    )
                )

                _last_openai_call_at = (
                    time.monotonic()
                )

                result = (
                    response.choices[0]
                    .message.content
                )

                # Save to cache
                _openai_cache[cache_key] = result

                return result

            except (
                RateLimitError,
                OpenAIError,
            ) as error:

                _last_openai_call_at = (
                    time.monotonic()
                )

                error_text = str(error)

                print(
                    f"OpenAI Error: {error_text}"
                )

                if (
                    "429" in error_text
                    or "503" in error_text
                    or "rate_limit" in error_text.lower()
                ):
                    sleep_for = (
                        _retry_delay_seconds(
                            error,
                            default=30,
                        )
                    )

                    print(
                        f"Retrying in "
                        f"{sleep_for:.0f} seconds..."
                    )

                    time.sleep(sleep_for)
                    continue

                raise

    raise RuntimeError(
        "OpenAI failed after maximum retries."
    )

RAG Evaluation

In [8]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\HP\AppData\Local\Temp\ipykernel_39596\1796813831.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
d:\VS Code projects\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [9]:
import os
import requests
import tempfile

from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings

from langchain_community.document_loaders import (
    PyPDFLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_core.vectorstores import (
    InMemoryVectorStore
)

#########################################################
# Load Environment Variables
#########################################################

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

#########################################################
# PDF URLs
#########################################################

urls = [
    "https://github.com/kunalarya873/PythonBooks/raw/main/Automate%20the%20Boring%20Stuff%20with%20Python.pdf",

    "https://github.com/ValeriiaMur/Python-books-and-exercises/raw/master/python-crash-course.pdf",

    "https://github.com/DanielRizvi/oreilly-books-collection-/raw/main/Data-Science-and-AI/LLMs/Hands-On-Large-Language-Models.pdf",

    "https://github.com/DanielRizvi/oreilly-books-collection-/raw/main/Data-Science-and-AI/AI/AI%20Engineering.pdf",

    "https://github.com/DanielRizvi/oreilly-books-collection-/raw/main/Data-Science-and-AI/3D-DataScience/3D-Data-Science-with-Python-Building-Accurate-Digital-Environments-with-3D-Point-Cloud-Workflows.pdf",
]

#########################################################
# Download and Load PDFs
#########################################################

docs = []

session = requests.Session()

for url in urls:
    try:
        print(f"\nDownloading:\n{url}")

        response = session.get(
            url,
            timeout=120,
        )
        response.raise_for_status()

        with tempfile.NamedTemporaryFile(
            suffix=".pdf",
            delete=False,
        ) as temp_pdf:

            temp_pdf.write(response.content)
            pdf_path = temp_pdf.name

        loader = PyPDFLoader(pdf_path)
        pdf_docs = loader.load()

        docs.extend(pdf_docs)

        print(
            f"Loaded {len(pdf_docs)} pages."
        )

        os.remove(pdf_path)

    except Exception as e:
        print(f"\nFailed:\n{url}")
        print(e)

#########################################################
# Split Documents
#########################################################

text_splitter = (
    RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
    )
)

docs_split = text_splitter.split_documents(
    docs
)

print(
    f"\nTotal Chunks: {len(docs_split)}"
)

#########################################################
# Embedding Model (OpenAI)
#########################################################

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY,
)

#########################################################
# Vector Store
#########################################################

vectorstore = (
    InMemoryVectorStore.from_documents(
        documents=docs_split,
        embedding=embeddings,
    )
)

#########################################################
# Retriever
#########################################################

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3
    }
)

print(
    "\nVector Store Created Successfully!"
)


Downloading:
https://github.com/kunalarya873/PythonBooks/raw/main/Automate%20the%20Boring%20Stuff%20with%20Python.pdf
Loaded 505 pages.

Downloading:
https://github.com/ValeriiaMur/Python-books-and-exercises/raw/master/python-crash-course.pdf
Loaded 562 pages.

Downloading:
https://github.com/DanielRizvi/oreilly-books-collection-/raw/main/Data-Science-and-AI/LLMs/Hands-On-Large-Language-Models.pdf
Loaded 598 pages.

Downloading:
https://github.com/DanielRizvi/oreilly-books-collection-/raw/main/Data-Science-and-AI/AI/AI%20Engineering.pdf
Loaded 535 pages.

Downloading:
https://github.com/DanielRizvi/oreilly-books-collection-/raw/main/Data-Science-and-AI/3D-DataScience/3D-Data-Science-with-Python-Building-Accurate-Digital-Environments-with-3D-Point-Cloud-Workflows.pdf
Loaded 957 pages.

Total Chunks: 12100

Vector Store Created Successfully!


In [10]:
docs = retriever.invoke(
    "What is LLMs?"
)


print(docs[0].page_content)

LLM applications are incredibly satisfying to create since they are partially
bounded by the things you can imagine. As these models grow more
accurate, using them in practice for creative use cases such as role-playing
and writing children’s books simply becomes more and more fun.
Responsible LLM Development and Usage
The impact of LLMs has been and likely continues to be significant due to
their widespread adoption. As we explore the incredible capabilities of


In [11]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0,
    max_completion_tokens=256,
)

llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, profile={'name': 'GPT-5 Mini', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000027A44FF70E0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000027A44FF7620>, root_client=<openai.OpenAI object 

In [12]:
from langsmith import traceable

_cache = {}

@traceable()
def rag_bot(question: str) -> dict:

    # Cache repeated questions
    if question in _cache:
        return _cache[question]

    docs = retriever.invoke(question)

    docs_string = "\n\n".join(
        doc.page_content[:1000]
        for doc in docs
    )

    instructions = f"""
Answer using only the context below.
If the answer is not present, say "I don't know".

Context:
{docs_string}
"""

    ai_msg = llm.invoke(
        [
            {
                "role": "system",
                "content": instructions,
            },
            {
                "role": "user",
                "content": question,
            },
        ]
    )

    result = {
        "answer": ai_msg.content,
        "documents": docs,
    }

    _cache[question] = result

    return result

Dataset

In [13]:
from langsmith import Client

client = Client()

examples = [
    {
        "inputs": {
            "question": "What is a Python list?"
        },
        "outputs": {
            "answer": "A Python list is an ordered, mutable collection of items that can store elements of different data types."
        }
    },
    {
        "inputs": {
            "question": "What is the purpose of a function in Python?"
        },
        "outputs": {
            "answer": "A function is a reusable block of code that performs a specific task and can accept inputs and return outputs."
        }
    },
    {
        "inputs": {
            "question": "What is web scraping in Python?"
        },
        "outputs": {
            "answer": "Web scraping is the process of automatically extracting data from websites using Python libraries such as BeautifulSoup and requests."
        }
    },
    {
        "inputs": {
            "question": "What is prompt engineering?"
        },
        "outputs": {
            "answer": "Prompt engineering is the practice of designing and optimizing prompts to improve the responses generated by large language models."
        }
    },
    {
        "inputs": {
            "question": "What is an embedding in large language models?"
        },
        "outputs": {
            "answer": "An embedding is a numerical vector representation of text that captures its semantic meaning and relationships."
        }
    },
    {
        "inputs": {
            "question": "What is Retrieval-Augmented Generation (RAG)?"
        },
        "outputs": {
            "answer": "Retrieval-Augmented Generation is a technique that combines information retrieval with language generation by supplying relevant documents as context to a language model."
        }
    },
    {
        "inputs": {
            "question": "What is a vector database?"
        },
        "outputs": {
            "answer": "A vector database stores high-dimensional embeddings and enables efficient similarity search for AI applications."
        }
    },
    {
        "inputs": {
            "question": "What is point cloud data in 3D data science?"
        },
        "outputs": {
            "answer": "Point cloud data is a collection of points in three-dimensional space representing the external surface of an object or environment."
        }
    },
    {
        "inputs": {
            "question": "What is supervised learning?"
        },
        "outputs": {
            "answer": "Supervised learning is a machine learning approach in which a model learns from labeled training data to make predictions."
        }
    },
    {
        "inputs": {
            "question": "What is tokenization in natural language processing?"
        },
        "outputs": {
            "answer": "Tokenization is the process of breaking text into smaller units such as words, subwords, or characters for further processing."
        }
    }
]

# Create dataset if it doesn't already exist
dataset_name = "RAG Test Evaluation"

try:
    dataset = client.create_dataset(
        dataset_name=dataset_name
    )
    print(f"Created dataset: {dataset_name}")

except Exception:
    dataset = client.read_dataset(
        dataset_name=dataset_name
    )
    print(f"Using existing dataset: {dataset_name}")

# Upload examples
client.create_examples(
    dataset_id=dataset.id,
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples]
)

print(f"Successfully uploaded {len(examples)} examples.")
print("Dataset ID:", dataset.id)

Using existing dataset: RAG Test Evaluation
Successfully uploaded 10 examples.
Dataset ID: 6014c5bf-d15d-40a9-bafa-ec9fb7a4214e


Evaluators

In [14]:
#Evaluators


In [15]:
from typing_extensions import (
    Annotated,
    TypedDict,
)

In [16]:
import os
from typing_extensions import Annotated, TypedDict
from langchain_openai import ChatOpenAI


############################################################
# Output Schema
############################################################

class CorrectnessGrade(TypedDict):
    explanation: Annotated[
        str,
        "Brief reason for the decision."
    ]

    correct: Annotated[
        bool,
        "True if correct, otherwise False."
    ]


############################################################
# Evaluation Instructions
############################################################

correctness_instructions = """
Evaluate whether the GENERATED ANSWER is factually equivalent
to the GROUND TRUTH ANSWER.

Rules:
1. True if both answers mean the same thing.
2. Extra information is allowed if correct.
3. False if incorrect, misleading, incomplete,
or contradictory.
4. False if the answer says 'I don't know'
when the ground truth contains the answer.

Return:
1. A short explanation.
2. The final boolean value.
"""


############################################################
# OpenAI Evaluator
############################################################

grader_llm = (
    ChatOpenAI(
        model="gpt-5-mini",
        api_key=os.getenv(
            "OPENAI_API_KEY"
        ),
        temperature=0,
        max_completion_tokens=64,
    )
    .with_structured_output(
        CorrectnessGrade
    )
)


############################################################
# Evaluator Function
############################################################

def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> bool:

    prompt = (
        f"Q:{inputs['question']}\n"
        f"GT:{reference_outputs['answer']}\n"
        f"A:{outputs['answer']}"
    )

    grade = grader_llm.invoke(
        [
            {
                "role": "system",
                "content": correctness_instructions,
            },
            {
                "role": "user",
                "content": prompt,
            },
        ]
    )

    print(
        "Correct:",
        grade["correct"]
    )

    return grade["correct"]

Relavence : Response vs Input

In [17]:
from typing_extensions import Annotated, TypedDict
from langchain_openai import ChatOpenAI
import os


############################################################
# Output Schema
############################################################

class RelevanceGrade(TypedDict):
    explanation: Annotated[
        str,
        "Brief reason for the decision."
    ]

    relevant: Annotated[
        bool,
        "True if relevant, otherwise False."
    ]


############################################################
# Relevance Instructions
############################################################

relevance_instructions = """
Determine whether the GENERATED ANSWER
addresses the USER QUESTION.

Rules:
1. True if the answer addresses the question.
2. Extra information is allowed if relevant.
3. False if off-topic, misleading, incomplete,
or unrelated.
4. False if the answer says
'I don't know' and the question can be answered.

Return:
1. A short explanation.
2. The final boolean value.
"""


############################################################
# OpenAI Evaluator Model
############################################################

relevance_llm = (
    ChatOpenAI(
        model="gpt-5-mini",
        api_key=os.getenv(
            "OPENAI_API_KEY"
        ),
        temperature=0,
        max_completion_tokens=64,
    )
    .with_structured_output(
        RelevanceGrade
    )
)


############################################################
# Evaluator
############################################################

def relevance(
    inputs: dict,
    outputs: dict,
) -> bool:

    prompt = (
        f"Q:{inputs['question']}\n"
        f"A:{outputs['answer']}"
    )

    grade = relevance_llm.invoke(
        [
            {
                "role": "system",
                "content": relevance_instructions,
            },
            {
                "role": "user",
                "content": prompt,
            },
        ]
    )

    print(
        "Relevant:",
        grade["relevant"]
    )

    return grade["relevant"]

Groundedness:Response vs Retrieved docs

In [18]:
from typing_extensions import Annotated, TypedDict
from langchain_openai import ChatOpenAI
import os


############################################################
# Output Schema
############################################################

class GroundedGrade(TypedDict):
    explanation: Annotated[
        str,
        "Brief reason for the decision."
    ]

    grounded: Annotated[
        bool,
        "True if supported by the context, otherwise False."
    ]


############################################################
# Instructions
############################################################

grounded_instructions = """
Determine whether the GENERATED ANSWER
is fully supported by the RETRIEVED CONTEXT.

Rules:
1. Every fact in the answer must be supported
   by the context.
2. Rewording and summarization are allowed.
3. If the answer contains unsupported information,
   return False.

Return:
1. A short explanation.
2. The final boolean value.
"""


############################################################
# OpenAI Evaluator
############################################################

grounded_llm = (
    ChatOpenAI(
        model="gpt-5-mini",
        api_key=os.getenv(
            "OPENAI_API_KEY"
        ),
        temperature=0,
        max_completion_tokens=64,
    )
    .with_structured_output(
        GroundedGrade
    )
)


############################################################
# Evaluator
############################################################

def groundedness(
    inputs: dict,
    outputs: dict,
) -> bool:

    # Limit context size to reduce tokens
    doc_string = "\n\n".join(
        doc.page_content[:1000]
        for doc in outputs["documents"]
    )

    prompt = (
        f"C:{doc_string}\n"
        f"A:{outputs['answer']}"
    )

    grade = grounded_llm.invoke(
        [
            {
                "role": "system",
                "content": grounded_instructions,
            },
            {
                "role": "user",
                "content": prompt,
            },
        ]
    )

    print(
        "Grounded:",
        grade["grounded"]
    )

    return grade["grounded"]

Retrieval Relevance: Retrieved docs vs Input

In [19]:
from typing_extensions import Annotated, TypedDict
from langchain_openai import ChatOpenAI
import os


############################################################
# Output Schema
############################################################

class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[
        str,
        "Brief reason for the decision."
    ]

    relevant: Annotated[
        bool,
        "True if relevant, otherwise False."
    ]


############################################################
# Instructions
############################################################

retrieval_relevance_instructions = """
Determine whether the RETRIEVED DOCUMENTS
contain information relevant to the USER QUESTION.

Rules:
1. True if the documents contain keywords,
   concepts, or semantic information related
   to the question.
2. Documents do not need to answer the entire
   question perfectly.
3. Some unrelated information is acceptable.
4. False only if the documents are completely
   unrelated.

Return:
1. A short explanation.
2. The final boolean value.
"""


############################################################
# OpenAI Evaluator
############################################################

retrieval_relevance_llm = (
    ChatOpenAI(
        model="gpt-5-mini",
        api_key=os.getenv(
            "OPENAI_API_KEY"
        ),
        temperature=0,
        max_completion_tokens=64,
    )
    .with_structured_output(
        RetrievalRelevanceGrade
    )
)


############################################################
# Evaluator
############################################################

def retrieval_relevance(
    inputs: dict,
    outputs: dict,
) -> bool:

    # Limit document size to reduce tokens
    doc_string = "\n\n".join(
        doc.page_content[:1000]
        for doc in outputs["documents"]
    )

    prompt = (
        f"Q:{inputs['question']}\n"
        f"D:{doc_string}"
    )

    grade = retrieval_relevance_llm.invoke(
        [
            {
                "role": "system",
                "content": retrieval_relevance_instructions,
            },
            {
                "role": "user",
                "content": prompt,
            },
        ]
    )

    print(
        "Relevant:",
        grade["relevant"]
    )

    return grade["relevant"]

Run the Evaluation

In [20]:
from langsmith import Client
import pandas as pd

client = Client()

dataset_name = "RAG Test Evaluation"

#########################################################
# Cache
#########################################################

_prediction_cache = {}

#########################################################
# Target Function
#########################################################

def target(inputs: dict) -> dict:
    question = inputs["question"]

    if question in _prediction_cache:
        return _prediction_cache[question]

    result = rag_bot(question)

    _prediction_cache[question] = result

    return result


#########################################################
# Correctness Evaluator
#########################################################

def correctness(run, example):
    prediction = (
        run.outputs.get("answer", "")
        if run.outputs
        else ""
    )

    reference = (
        example.outputs.get("answer", "")
        if example.outputs
        else ""
    )

    score = float(
        prediction.strip().lower()
        == reference.strip().lower()
    )

    return {
        "key": "correctness",
        "score": score,
    }


#########################################################
# Relevance Evaluator
#########################################################

def relevance(run, example):
    prediction = (
        run.outputs.get("answer", "")
        if run.outputs
        else ""
    )

    question = (
        example.inputs.get("question", "")
        if example.inputs
        else ""
    )

    words = question.lower().split()

    relevant = any(
        word in prediction.lower()
        for word in words
    )

    return {
        "key": "relevance",
        "score": float(relevant),
    }


#########################################################
# Groundedness Evaluator
#########################################################

def groundedness(run, example):
    docs = (
        run.outputs.get("documents", [])
        if run.outputs
        else []
    )

    answer = (
        run.outputs.get("answer", "")
        if run.outputs
        else ""
    )

    if len(docs) == 0:
        score = 0.0
    else:
        context = " ".join(
            doc.page_content[:500]
            for doc in docs
        ).lower()

        score = float(
            answer.lower()[:100] in context
        )

    return {
        "key": "groundedness",
        "score": score,
    }


#########################################################
# Retrieval Relevance Evaluator
#########################################################

def retrieval_relevance(run, example):
    docs = (
        run.outputs.get("documents", [])
        if run.outputs
        else []
    )

    question = (
        example.inputs.get("question", "")
        if example.inputs
        else ""
    )

    context = " ".join(
        doc.page_content[:500]
        for doc in docs
    ).lower()

    words = question.lower().split()

    relevant = any(
        word in context
        for word in words
    )

    return {
        "key": "retrieval_relevance",
        "score": float(relevant),
    }


#########################################################
# Run Evaluation
#########################################################

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[
        correctness,
        relevance,
        groundedness,
        retrieval_relevance,
    ],
    experiment_prefix="openai-rag-evaluation",
    metadata={
        "llm": "gpt-5-mini",
        "application": "RAG over AI-ML-LLM-Python books",
        "version": "v1",
    },
)

#########################################################
# Results
#########################################################

df = experiment_results.to_pandas()

print(df.head())

View the evaluation results for experiment: 'openai-rag-evaluation-4e231516' at:
https://smith.langchain.com/o/e286cd95-3783-40df-adc3-7aa394763f9a/datasets/6014c5bf-d15d-40a9-bafa-ec9fb7a4214e/compare?selectedSessions=1004bc6d-1adb-416c-9ce5-0b51c5fa939f




40it [01:02,  1.57s/it]

                                 inputs.question  \
0   What is point cloud data in 3D data science?   
1  What is Retrieval-Augmented Generation (RAG)?   
2   What is the purpose of a function in Python?   
3                         What is a Python list?   
4                     What is a vector database?   

                                      outputs.answer  \
0                                                      
1                                                      
2                                                      
3                                                      
4  A vector database is a data store for vectors ...   

                                   outputs.documents error  \
0  [page_content='A point cloud comprises a set o...  None   
1  [page_content='R\nRAG (retrieval-augmented gen...  None   
2  [page_content='the function dedicated to handl...  None   
3  [page_content='list without changing other lin...  None   
4  [page_content='databases have extende